# Tien Xu Ly, Bien Doi & EDA Chi Tiet -- Online Retail II (Cam Tay Chi Viec)

Notebook nay trien khai chi tiet **Phan 6 (EDA)** va **Phan 7 (Data Preprocessing)** trong file ly thuyet (`../02-ly-thuyet/nen-tang-kien-thuc-explainable-ml-churn.md`), va bo sung phan **Bien doi du lieu (Transformation)** de chuan bi cho Feature Engineering o notebook `thuc-hanh-churn-prediction.ipynb`.

Thu tu logic: **EDA truoc** (de phat hien van de) -> **Tien xu ly** (de sua cac van de da phat hien) -> **Bien doi** (de chuan bi dung dinh dang cho mo hinh). Moi buoc deu giai thich **tai sao lam** va **can nhin gi trong ket qua** truoc khi chay code.

**Truoc khi chay**: dat file `online_retail_II.csv` vao thu muc `data/` cung cap voi notebook nay (xem `../04-cong-cu/huong-dan-cai-dat-cong-cu.md` Buoc 8).

## Muc luc

**PHAN A -- EDA (Kham pha du lieu)**
1. Nap du lieu & tong quan
2. Kiem tra chat luong du lieu (missing, duplicate)
3. Phan tich rieng: hoa don huy & ma khong phai san pham
4. Phan tich don bien (Quantity, Price, Country)
5. Phan tich theo thoi gian (tinh thoi vu)
6. Phan tich o muc khach hang
7. Phan tich o muc san pham
8. Phan tich song bien
9. Phan tich da bien / ma tran tuong quan
10. Phat hien outlier

**PHAN B -- Tien xu ly du lieu (Preprocessing)**
11. Xay dung pipeline lam sach chinh thuc
12. Xu ly outlier (capping)

**PHAN C -- Bien doi du lieu (Transformation)**
13. Encoding bien phan loai (Country)
14. Log-transform & chuan hoa (scaling) cac dac trung lech phai
15. Luu du lieu da xu ly -> chuan bi cho Feature Engineering
16. Tong ket & buoc tiep theo

---
# PHAN A -- EDA (Kham Pha Du Lieu)

## Buoc 1 -- Nap du lieu & tong quan

**Tai sao lam**: can nap dung kieu du lieu (dac biet cot ngay gio) truoc khi phan tich bat cu dieu gi.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)


In [ ]:
df = pd.read_csv('data/online_retail_II.csv', encoding='ISO-8859-1')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"Kich thuoc du lieu: {df.shape[0]:,} dong x {df.shape[1]} cot")
df.head()


**Can nhin gi**: kiem tra ten cot dung nhu mo ta trong `mo-ta-du-lieu-online-retail-ii.md` (Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country). Neu file ban tai la ban .xlsx voi 2 sheet, xem lai notebook `thuc-hanh-churn-prediction.ipynb` phan nap du lieu de biet cach gop 2 sheet.

## Buoc 2 -- Kiem tra chat luong du lieu

**Tai sao lam**: truoc khi ve bieu do gi, can biet du lieu thieu o dau va co dong trung lap khong -- day la co so de quyet dinh cach tien xu ly o Phan B.

In [ ]:
df.info()


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'So dong thieu': missing, 'Ty le %': missing_pct}).sort_values('Ty le %', ascending=False)


**Can nhin gi**: cot Customer ID thuong thieu mot ty le dang ke -- day la cac giao dich khong dinh danh duoc khach hang, se can quyet dinh xu ly o Buoc 11.

In [ ]:
n_duplicates = df.duplicated().sum()
print(f"So dong trung lap hoan toan: {n_duplicates:,} ({n_duplicates/len(df)*100:.2f}%)")


**Can nhin gi**: mot ty le nho trung lap la binh thuong -- ghi nhan lai de xu ly chinh thuc o Buoc 11.

## Buoc 3 -- Phan tich rieng: hoa don huy & ma khong phai san pham

**Tai sao lam**: day la dac diem nghiep vu quan trong nhat cua bo du lieu nay -- can do luong TRUOC khi loai bo, de biet chung chiem bao nhieu % va co xu huong gi theo thoi gian khong.

In [ ]:
df['is_cancelled'] = df['Invoice'].astype(str).str.startswith('C')
cancel_rate = df['is_cancelled'].mean() * 100
print(f"Ty le dong la hoa don huy: {cancel_rate:.2f}%")

cancel_by_month = df.set_index('InvoiceDate').resample('ME')['is_cancelled'].mean() * 100
cancel_by_month.plot(title='Ty le hoa don huy theo thang (%)', marker='o')
plt.ylabel('% huy')
plt.show()


**Can nhin gi**: neu ty le huy on dinh quanh mot muc (vi du 1-2%) suot 2 nam, do la muc nen binh thuong cua van hanh. Neu co thang dot bien, ghi chu lai de giai thich trong phan Discussion sau nay.

In [ ]:
non_product_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'DOT', 'ADJUST', 'ADJUST2', 'CRUK']
mask_non_product = df['StockCode'].astype(str).isin(non_product_codes)
print(f"So dong la ma khong phai san pham: {mask_non_product.sum():,} ({mask_non_product.sum()/len(df)*100:.3f}%)")
df.loc[mask_non_product, 'StockCode'].value_counts()


**Can nhin gi**: thuong chiem ty le rat nho nhung van can loai vi chung khong phai hanh vi mua sam that -- neu de lai se lam meo cac chi so RFM sau nay.

## Buoc 4 -- Phan tich don bien (Univariate Analysis)

**Tai sao lam**: hieu hinh dang phan phoi tung bien so truoc khi tinh toan bat ky dac trung nao -- day la Phan 6.1 trong file ly thuyet. Tao ban du lieu da loc huy don + ma khong san pham de phan tich tiep.

In [ ]:
df_clean = df[~df['is_cancelled'] & ~mask_non_product].copy()
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']
print(f"Con lai: {len(df_clean):,} dong ({len(df_clean)/len(df)*100:.1f}% du lieu goc)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_clean['Quantity'], bins=50, ax=axes[0])
axes[0].set_title('Phan phoi Quantity (toan bo, chua loc outlier)')
sns.histplot(df_clean[df_clean['Quantity'].between(0, 50)]['Quantity'], bins=50, ax=axes[1])
axes[1].set_title('Phan phoi Quantity (chi xem 0-50, vung pho bien)')
plt.tight_layout()
plt.show()


**Can nhin gi**: bieu do ben trai gan nhu chi thay mot cot sat truc 0 vi co vai don ban buon voi Quantity cuc lon keo dai truc X -- day la bang chung truc quan cho viec khach ban buon tron khach le. Bieu do ben phai moi thay ro da so don hang mua voi so luong nho (duoi 12 san pham/dong).

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df_clean['Price'])
plt.title('Boxplot Price - phat hien outlier gia')
plt.show()
print(df_clean['Price'].describe())


**Can nhin gi**: neu thay vai diem gia tri Price cuc lon tach biet han khoi phan lon du lieu, do la ung vien outlier can xem xet o Buoc 10 -- co the la san pham cao cap that hoac loi nhap lieu, can doi chieu voi cot Description de phan biet.

In [ ]:
country_counts = df_clean['Country'].value_counts().head(10)
plt.figure(figsize=(10, 5))
sns.barplot(x=country_counts.values, y=country_counts.index, orient='h')
plt.title('Top 10 quoc gia theo so dong giao dich')
plt.xlabel('So dong')
plt.show()


**Can nhin gi**: United Kingdom gan nhu chac chan chiem ap dao (thuong tren 80-90% so dong) -- dieu nay anh huong truc tiep den cach xu ly cot Country o Buoc 13 (One-Hot Encoding se tao ra nhieu cot cuc ky thua cho cac quoc gia co rat it giao dich).

## Buoc 5 -- Phan tich theo thoi gian (tinh thoi vu)

**Tai sao lam**: cong ty ban qua tang nen kha nang cao co tinh thoi vu manh -- can xac nhan bang du lieu de tranh chon moc cutoff ngay truoc mua cao diem khi gan nhan churn.

In [ ]:
monthly_revenue = df_clean.set_index('InvoiceDate').resample('ME')['Revenue'].sum()
monthly_revenue.plot(marker='o', title='Doanh thu theo thang')
plt.ylabel('Revenue')
plt.show()


**Can nhin gi**: neu thay doanh thu tang vot vao thang 10-11 (chuan bi Giang sinh) roi giam manh vao thang 1-2, do xac nhan tinh thoi vu -- can can nhac khi chon nhieu moc cutoff cho sensitivity analysis (Phan 3.3 trong file ly thuyet).

In [ ]:
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
order_days = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
sns.countplot(x='DayOfWeek', data=df_clean, order=order_days, ax=axes[0])
axes[0].set_title('So giao dich theo Thu trong tuan')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(x='Hour', data=df_clean, ax=axes[1])
axes[1].set_title('So giao dich theo Gio trong ngay')
plt.tight_layout()
plt.show()


**Can nhin gi**: vi day la B2B/B2C tron lan voi nhieu khach ban buon, thuong se thay giao dich tap trung vao gio hanh chinh (9h-16h) cac ngay trong tuan, gan nhu khong co giao dich cuoi tuan.

## Buoc 6 -- Phan tich o muc khach hang

**Tai sao lam**: bai toan churn duoc xay tren du lieu theo khach hang, khong phai theo dong giao dich -- can hieu phan phoi hanh vi khach hang truoc khi tinh RFM chinh thuc.

In [ ]:
df_customer = df_clean.dropna(subset=['Customer ID'])
customer_summary = df_customer.groupby('Customer ID').agg(
    n_orders=('Invoice', 'nunique'),
    total_spend=('Revenue', 'sum'),
    n_countries=('Country', 'nunique'),
)
print(f"So khach hang duy nhat: {len(customer_summary):,}")
customer_summary.describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(customer_summary['n_orders'].clip(upper=50), bins=50, ax=axes[0])
axes[0].set_title('So don hang / khach hang (da cat o 50)')

sns.histplot(np.log1p(customer_summary['total_spend']), bins=50, ax=axes[1])
axes[1].set_title('log(1 + Tong chi tieu) / khach hang')
plt.tight_layout()
plt.show()


**Can nhin gi**: ca hai phan phoi gan nhu chac chan lech phai manh -- da so khach hang chi mua vai don voi chi tieu vua phai, nhung mot so it khach ban buon mua rat nhieu lan voi tong chi tieu rat lon. Day la ly do can log-transform o Buoc 14.

In [ ]:
top10_customers = customer_summary.sort_values('total_spend', ascending=False).head(10)
top10_customers


**Can nhin gi**: nhom Top 10 khach hang nay thuong dong gop mot ty le doanh thu khong tuong xung voi so luong -- day chinh la nhom khach hang CLV cao (Phan 3.4 trong file ly thuyet).

In [ ]:
reference_date = df_customer['InvoiceDate'].max()
recency = df_customer.groupby('Customer ID')['InvoiceDate'].max().apply(lambda x: (reference_date - x).days)
plt.figure(figsize=(10, 4))
sns.histplot(recency, bins=50)
plt.title(f'Phan phoi Recency (so ngay tinh den {reference_date.date()})')
plt.xlabel('So ngay')
plt.show()


**Can nhin gi**: neu phan phoi nay co 2 cum ro ret -- mot cum gan 0 (khach vua mua gan day) va mot cum rai dai ve phia cac gia tri lon (khach lau khong quay lai) -- day chinh la tin hieu truc quan som cho thay Recency se la dac trung phan biet churn rat manh.

## Buoc 7 -- Phan tich o muc san pham

**Tai sao lam**: hieu san pham nao ban chay giup dien giai insight sau nay.

In [ ]:
top_products_qty = df_clean.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
top_products_rev = df_clean.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=top_products_qty.values, y=top_products_qty.index, orient='h', ax=axes[0])
axes[0].set_title('Top 10 san pham theo SO LUONG ban ra')
sns.barplot(x=top_products_rev.values, y=top_products_rev.index, orient='h', ax=axes[1])
axes[1].set_title('Top 10 san pham theo DOANH THU')
plt.tight_layout()
plt.show()


**Can nhin gi**: hai bang xep hang nay thuong khac nhau -- san pham ban so luong nhieu nhat chua chac mang lai doanh thu cao nhat. Ghi nhan su khac biet nay de giai thich dung ngu canh khi viet Discussion.

## Buoc 8 -- Phan tich song bien (Bivariate Analysis)

**Tai sao lam**: xem moi quan he giua 2 bien cung luc -- Phan 6.2 trong file ly thuyet, dac biet quan trong de phat hien da cong tuyen truoc khi dua vao mo hinh.

In [ ]:
plt.figure(figsize=(8, 6))
sample = customer_summary.sample(min(2000, len(customer_summary)), random_state=42)
sns.scatterplot(x='n_orders', y='total_spend', data=sample, alpha=0.4)
plt.yscale('log')
plt.title('Quan he giua So don hang va Tong chi tieu (truc Y dang log)')
plt.show()


**Can nhin gi**: neu thay xu huong tang dan ro ret -- day la dau hieu Frequency va Monetary tuong quan cao, can kiem tra ky o Buoc 9.

In [ ]:
top5_countries = df_clean['Country'].value_counts().head(5).index
subset = df_clean[df_clean['Country'].isin(top5_countries)]
plt.figure(figsize=(10, 5))
sns.boxplot(x='Country', y='Revenue', data=subset[subset['Revenue'].between(0, subset['Revenue'].quantile(0.95))])
plt.title('Phan phoi Revenue theo Quoc gia (top 5, da cat outlier 5% cao nhat)')
plt.show()


**Can nhin gi**: neu mot quoc gia co Revenue trung vi cao han han so voi cac nuoc con lai, co the day la thi truong co khach ban buon tap trung.

## Buoc 9 -- Phan tich da bien / Ma tran tuong quan

**Tai sao lam**: Phan 6.3 trong file ly thuyet -- bat buoc phai lam truoc feature engineering chinh thuc, de quyet dinh co can loai bot/ket hop dac trung nao khong.

In [ ]:
rfm_preview = df_customer.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('Revenue', 'sum'),
)
rfm_preview['AvgOrderValue'] = rfm_preview['Monetary'] / rfm_preview['Frequency']

plt.figure(figsize=(7, 5))
sns.heatmap(rfm_preview.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Ma tran tuong quan cac dac trung RFM')
plt.show()


**Can nhin gi**: neu he so tuong quan giua Frequency va Monetary lon hon khoang 0.7-0.8, day la da cong tuyen dang ke -- can giai thich ro cach xu ly trong Methodology.

## Buoc 10 -- Phat hien Outlier

**Tai sao lam**: Phan 7.2 trong file ly thuyet -- can phan biet outlier that (hanh vi ban buon co that) va outlier loi (sai sot nhap lieu) truoc khi quyet dinh giu/loai o Phan B.

In [ ]:
q99 = df_clean['Quantity'].quantile(0.99)
q1 = df_clean['Quantity'].quantile(0.01)
print(f"1% thap nhat: {q1}, 99% cao nhat: {q99}")
print(f"So dong Quantity > 1000 (nghi ngo ban buon lon): {(df_clean['Quantity'] > 1000).sum():,}")

df_clean.nlargest(10, 'Quantity')[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'Country']]


**Can nhin gi**: nhin cot Description cua cac dong Quantity lon nhat -- neu la san pham hop ly (vi du do trang tri gia re so luong lon) thi day la outlier that (khach ban buon), nen giu lai hoac xu ly bang capping thay vi xoa thang. Neu thay Description trong hoac bat thuong, do co the la loi nhap lieu, nen xem xet loai bo.

**Quyet dinh goi y cho de tai nay**: giu lai cac don ban buon hop le, nhung ap dung capping/log-transform cho Monetary/Quantity khi dung mo hinh tuyen tinh -- se lam chi tiet o Buoc 12 va 14.

---
# PHAN B -- Tien Xu Ly Du Lieu (Preprocessing)

## Buoc 11 -- Xay dung pipeline lam sach chinh thuc

**Tai sao lam**: sau khi da EDA va biet ro tung van de nam o dau, gio dong goi lai thanh MOT ham lam sach duy nhat -- de dung lai nhat quan o notebook Feature Engineering, tranh lap lai code rai rac.

In [ ]:
def lam_sach_du_lieu(df_raw):
    """
    Pipeline lam sach chinh thuc cho Online Retail II.
    Dua tren cac phat hien tu EDA o Phan A phia tren.
    """
    df = df_raw.copy()

    # 1. Loai hoa don huy (Buoc 3)
    df = df[~df['Invoice'].astype(str).str.startswith('C')]

    # 2. Loai ma khong phai san pham (Buoc 3)
    non_product_codes = ['POST', 'D', 'M', 'BANK CHARGES', 'DOT', 'ADJUST', 'ADJUST2', 'CRUK']
    df = df[~df['StockCode'].astype(str).isin(non_product_codes)]

    # 3. Loai dong thieu Customer ID (Buoc 2) -- bat buoc cho phan tich theo khach hang
    df = df.dropna(subset=['Customer ID'])

    # 4. Loai Quantity hoac Price <= 0 (giao dich khong hop le)
    df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

    # 5. Loai dong trung lap hoan toan (Buoc 2)
    df = df.drop_duplicates()

    # 6. Tinh Revenue
    df['Revenue'] = df['Quantity'] * df['Price']

    return df.reset_index(drop=True)

df_processed = lam_sach_du_lieu(df)
print(f"Truoc lam sach: {len(df):,} dong")
print(f"Sau lam sach : {len(df_processed):,} dong ({len(df_processed)/len(df)*100:.1f}%)")


**Can nhin gi**: so dong con lai sau khi lam sach nen khop voi con so da tinh rai rac o Phan A (Buoc 3 va Buoc 6) -- neu lech nhieu, kiem tra lai thu tu cac buoc loc trong ham tren.

## Buoc 12 -- Xu ly Outlier (Capping)

**Tai sao lam**: da xac dinh o Buoc 10 rang cac don ban buon la hanh vi that, nen KHONG xoa, chi **capping** (gioi han) de giam anh huong len cac mo hinh nhay cam voi outlier (dac biet Logistic Regression), ma van giu duoc thong tin la khach do co Quantity/Revenue lon.

In [ ]:
def cap_outlier(series, lower_q=0.01, upper_q=0.99):
    """Gioi han gia tri o percentile duoi/tren, khong xoa dong du lieu."""
    lower = series.quantile(lower_q)
    upper = series.quantile(upper_q)
    return series.clip(lower=lower, upper=upper)

df_processed['Quantity_capped'] = cap_outlier(df_processed['Quantity'])
df_processed['Revenue_capped'] = cap_outlier(df_processed['Revenue'])

print("Truoc capping - Quantity max:", df_processed['Quantity'].max())
print("Sau capping   - Quantity max:", df_processed['Quantity_capped'].max())


**Can nhin gi**: gia tri max sau capping se giam ro ret so voi truoc -- day chinh la muc dich, giu nguyen SO DONG du lieu (khong mat thong tin ve so luong giao dich) nhung gioi han do lon cua gia tri outlier.

---
# PHAN C -- Bien Doi Du Lieu (Transformation)

## Buoc 13 -- Encoding bien phan loai (Country)

**Tai sao lam**: hau het mo hinh ML yeu cau dau vao la so, can chuyen cot Country (dang chu) sang dang so -- da nhac trong Phan 5.2 cua file ly thuyet.

In [ ]:
# Vi UK chiem ap dao (Buoc 4), gom cac quoc gia it giao dich vao nhom 'Other'
# de tranh One-Hot Encoding tao ra qua nhieu cot cuc ky thua
top_countries = df_processed['Country'].value_counts().head(8).index
df_processed['Country_grouped'] = df_processed['Country'].where(
    df_processed['Country'].isin(top_countries), 'Other'
)

encoder = OneHotEncoder(sparse_output=False, drop='first')
country_encoded = encoder.fit_transform(df_processed[['Country_grouped']])
country_encoded_df = pd.DataFrame(
    country_encoded,
    columns=encoder.get_feature_names_out(['Country_grouped']),
    index=df_processed.index
)
print(f"So cot sau encoding: {country_encoded_df.shape[1]}")
country_encoded_df.head()


**Can nhin gi**: neu khong gom nhom 'Other' truoc, so cot sau One-Hot Encoding se bang so quoc gia duy nhat trong du lieu (co the vai chuc cot), da so cac cot do gan nhu toan gia tri 0 -- gom nhom giup giam nhieu ma khong mat nhieu thong tin huu ich.

## Buoc 14 -- Log-transform & Chuan hoa (Scaling)

**Tai sao lam**: da xac nhan o Buoc 6 rang Monetary/Frequency lech phai manh -- Logistic Regression (khong nhu tree-based models) nhay cam voi thang do va do lech cua du lieu dau vao, nen can xu ly truoc khi dua vao mo hinh nay.

In [ ]:
rfm_demo = rfm_preview.copy()

# Log-transform cho cac cot lech phai (cong 1 de tranh log(0))
rfm_demo['Monetary_log'] = np.log1p(rfm_demo['Monetary'])
rfm_demo['Frequency_log'] = np.log1p(rfm_demo['Frequency'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(rfm_demo['Monetary'], bins=50, ax=axes[0])
axes[0].set_title('Monetary - TRUOC log-transform (lech phai manh)')
sns.histplot(rfm_demo['Monetary_log'], bins=50, ax=axes[1])
axes[1].set_title('Monetary - SAU log-transform (gan doi xung hon)')
plt.tight_layout()
plt.show()


**Can nhin gi**: bieu do sau log-transform (ben phai) se can doi va gan voi hinh chuong hon nhieu so voi truoc -- day la dieu kien ly tuong hon cho cac mo hinh tuyen tinh, va cung giup viec ve bieu do/doc du lieu de nhin hon.

In [ ]:
# Chuan hoa (StandardScaler): dua ve trung binh 0, do lech chuan 1
# CHI fit tren tap TRAIN trong thuc te (tranh data leakage) -- o day minh hoa tren toan bo de xem hinh dang
scaler = StandardScaler()
features_to_scale = ['Recency', 'Frequency_log', 'Monetary_log']
rfm_demo[['Monetary_log']] = rfm_demo[['Monetary_log']]  # da co san o tren
scaled_values = scaler.fit_transform(rfm_demo[['Recency', 'Frequency_log', 'Monetary_log']])
rfm_scaled = pd.DataFrame(scaled_values, columns=[c + '_scaled' for c in features_to_scale], index=rfm_demo.index)
rfm_scaled.describe()


**Can nhin gi**: sau StandardScaler, moi cot se co trung binh xap xi 0 va do lech chuan xap xi 1 -- kiem tra dong 'mean' va 'std' trong bang tren de xac nhan. **Luu y quan trong**: trong pipeline thuc te (notebook Feature Engineering), buoc `scaler.fit()` chi duoc goi tren tap TRAIN, sau do dung `scaler.transform()` (khong fit lai) cho tap TEST -- fit tren toan bo du lieu nhu o day chi de minh hoa hinh dang phan phoi, neu lam vay trong mo hinh that se gay data leakage (Phan 8.3 trong file ly thuyet).

## Buoc 15 -- Luu du lieu da xu ly

**Tai sao lam**: luu lai ket qua Phan B (`df_processed`) thanh file rieng, de notebook Feature Engineering (`thuc-hanh-churn-prediction.ipynb`) co the doc thang vao ma khong phai chay lai toan bo EDA moi lan.

In [ ]:
df_processed.to_csv('data/online_retail_ii_processed.csv', index=False)
print(f"Da luu {len(df_processed):,} dong vao data/online_retail_ii_processed.csv")


---
## Buoc 16 -- Tong ket & buoc tiep theo

Checklist cac quyet dinh da chot qua EDA + Tien xu ly + Bien doi:

- [ ] Ty le hoa don huy va ma khong san pham -- da do luong va loai bo (Buoc 3, 11).
- [ ] Missing Customer ID -- da loai khoi phan tich theo khach hang (Buoc 2, 11).
- [ ] Outlier Quantity/Revenue -- xac dinh la hanh vi ban buon that, xu ly bang capping thay vi xoa (Buoc 10, 12).
- [ ] Tinh thoi vu ro ret theo thang -- can nhac khi chon nhieu moc cutoff cho gan nhan churn (Buoc 5).
- [ ] Frequency & Monetary tuong quan cao -- can giai thich cach xu ly trong Methodology (Buoc 9).
- [ ] Country -- da gom nhom truoc khi One-Hot Encoding de tranh qua thua (Buoc 13).
- [ ] Monetary/Frequency lech phai -- da log-transform truoc khi chuan hoa cho mo hinh tuyen tinh (Buoc 14).

**Buoc tiep theo**: mo `thuc-hanh-churn-prediction.ipynb`, doc du lieu tu `data/online_retail_ii_processed.csv` (thay vi file goc) de tiep tuc gan nhan churn va Feature Engineering chinh thuc -- moi quyet dinh tien xu ly/bien doi o day da duoc ap dung san.